In [1]:
# Importar las librerías necesarias
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [2]:
# Cargar las matrices de características de entrenamiento y prueba
X_train = pd.read_parquet('../data/X_train.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/y_train.parquet', engine='fastparquet')['is_fraud']
y_test = pd.read_parquet('../data/y_test.parquet', engine='fastparquet')['is_fraud']
    
# Verificar las dimensiones de los conjuntos de datos
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

X_train shape: (1296675, 19) | X_test shape: (555719, 19)


In [3]:
# Escalado de características: la Regresión Logística es sensible a las diferencias de escala
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# Inicializar Regresión Logística usando 'class_weight=balanced' para tratar el desbalance de clases
logic_model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Entrenar el modelo base (baseline)
logic_model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [5]:
# Generar predicciones en el conjunto de prueba
y_pred = logic_model.predict(X_test_scaled)

# Mostrar la matriz de confusión y el reporte de clasificación
print("=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===
[[486296  67278]
 [   556   1589]]

=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

           0       1.00      0.88      0.93    553574
           1       0.02      0.74      0.04      2145

    accuracy                           0.88    555719
   macro avg       0.51      0.81      0.49    555719
weighted avg       1.00      0.88      0.93    555719



**Evaluación de la Regresión Logística**: La precisión y el F1-Score para la clase positiva (fraude) resultaron insuficientes para un entorno de producción. Se cambia la estrategia a XGBoost para capturar fronteras de decisión no lineales.

In [6]:
# Calcular el factor de peso de clases para compensar el desbalance (Casos Negativos / Casos Positivos)
class_counts = y_train.value_counts()
scale_weight = class_counts[0] / class_counts[1]

# Inicializar XGBoost con configuración para balanceo de clases
xgb_model = XGBClassifier(
    n_estimators=300, 
    max_depth=5, 
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

In [7]:
# Validación cruzada con k-folds

# Queremos probar distintos umbrales para luego elegir el mejor en base a f1-scores
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

pr_auc_scores = []
# Para f1-scores usamos una lista ya que queremos guardar cada métrica de cada fold de cada umbral
f1_scores = {threshold: [] for threshold in thresholds}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# train_idx: índices de las filas para entrenar y val_idx índices de las filas para validar.
for train_idx, val_idx in skf.split(X_train, y_train):

    # Separar datos de entrenamiento y validación
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Entrenar XGBoost en el fold actual
    xgb_model.fit(X_tr, y_tr)

    # Obtener probabilidades de fraude      
    probs = xgb_model.predict_proba(X_val)[:, 1]

    # PR-AUC
    pr_auc_scores.append(
        average_precision_score(y_val, probs)
    )

    # Evaluar F1 para cada threshold
    for threshold in thresholds:
        # Convertimos la probabilidad en 0 o 1 (dependerá del threshold)
        preds = (probs >= threshold).astype(int)

        f1_scores[threshold].append(
            f1_score(y_val, preds)
        )


# Mostrar resultados promedio
print("=== CROSS-VALIDATION RESULTS (5-FOLD) ===")

print(
    f"Mean PR-AUC: "
    f"{np.mean(pr_auc_scores):.4f} ± {np.std(pr_auc_scores):.4f}"
)

# Vamos a mostrar los f1-scores de cada threshold
for threshold in thresholds:
    print(
        f"F1 (Threshold {threshold:.2f}): "
        f"{np.mean(f1_scores[threshold]):.4f} ± "
        f"{np.std(f1_scores[threshold]):.4f}"
    )


# Seleccionar el threshold con mayor F1 promedio
mean_f1 = {
    threshold: np.mean(f1_scores[threshold])
    for threshold in thresholds
}

best_threshold = max(mean_f1, key=mean_f1.get)

print("\n=== BEST THRESHOLD ===")
print(f"Threshold seleccionado: {best_threshold:.2f}")

=== CROSS-VALIDATION RESULTS (5-FOLD) ===
Mean PR-AUC: 0.9125 ± 0.0028
F1 (Threshold 0.50): 0.6439 ± 0.0032
F1 (Threshold 0.55): 0.6608 ± 0.0046
F1 (Threshold 0.60): 0.6780 ± 0.0064
F1 (Threshold 0.65): 0.6951 ± 0.0047
F1 (Threshold 0.70): 0.7123 ± 0.0048
F1 (Threshold 0.75): 0.7297 ± 0.0040
F1 (Threshold 0.80): 0.7491 ± 0.0047

=== BEST THRESHOLD ===
Threshold seleccionado: 0.80


In [8]:
# Entrenar el modelo final con todos los datos de entrenamiento
xgb_model.fit(X_train, y_train)

# Obtener probabilidades de fraude en el conjunto de test
y_probs_test = xgb_model.predict_proba(X_test)[:, 1]

# Aplicar el threshold seleccionado durante la validación
y_pred_test = (y_probs_test >= best_threshold).astype(int)

# Evaluación final sobre el conjunto de test
print("\n=== FINAL TEST RESULTS ===")
print(f"Threshold: {best_threshold:.2f}")

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred_test))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_test))


=== FINAL TEST RESULTS ===
Threshold: 0.80

CONFUSION MATRIX:
[[551782   1792]
 [   231   1914]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.52      0.89      0.65      2145

    accuracy                           1.00    555719
   macro avg       0.76      0.94      0.83    555719
weighted avg       1.00      1.00      1.00    555719



In [9]:
# Crear una copia de X_test para el dashboard
df_dashboard = X_test.copy()

# Agregar la etiqueta real
df_dashboard['is_fraud_real'] = y_test.values

# Agregar las probabilidades de fraude predichas por XGBoost
df_dashboard['fraud_probability'] = y_probs_test

# Agregar las predicciones usando el threshold seleccionado
df_dashboard['model_prediction'] = y_pred_test

# Agregar el threshold utilizado por el modelo
df_dashboard['threshold'] = best_threshold

# Agregar un indicador de error
# 1 = predicción incorrecta, 0 = predicción correcta
df_dashboard['prediction_error'] = (
    df_dashboard['is_fraud_real'] != df_dashboard['model_prediction']
).astype(int)

# Exportar el dataframe para Power BI
df_dashboard.to_csv(
    '../data/fraud_results_powerbi.csv',
    index=False
)

print("File 'fraud_results_powerbi.csv' generated successfully!")
print(f"Total records exported for dashboarding: {len(df_dashboard)}")

File 'fraud_results_powerbi.csv' generated successfully!
Total records exported for dashboarding: 555719


### Justificación del Modelo y Estrategia de Decisión

**Modelo Base con Regresión Logística:**
Se utilizó como punto de comparación (benchmark). Aunque `class_weight='balanced'` mejoró la detección de fraudes, el modelo generó demasiados falsos positivos, resultando en una precisión y F1-Score muy bajos para la clase positiva.

**Ventajas de XGBoost y Gradient Boosting:**
XGBoost permite capturar relaciones complejas y no lineales entre las variables. Además, `scale_pos_weight` permite compensar el fuerte desbalance entre transacciones legítimas y fraudulentas.

**Optimización del Umbral de Decisión:**
En lugar de utilizar el umbral estándar de 0.50, se evaluaron distintos umbrales mediante validación cruzada. El mejor resultado se obtuvo con un threshold de 0.80, mejorando el equilibrio entre precisión y recall.

**Validación Cruzada Estratificada (Stratified K-Fold):**
Debido al fuerte desbalance de clases (~0.52% de casos positivos), se utilizó StratifiedKFold con 5 particiones para mantener una proporción similar de clases en cada fold y comprobar la estabilidad del modelo mediante PR-AUC y F1-Score.

**Evaluación Final:**
Una vez seleccionado el modelo y el threshold, XGBoost se reentrenó utilizando todos los datos de entrenamiento y se evaluó sobre el conjunto de test, reservado para medir el rendimiento final sobre datos no utilizados durante el entrenamiento ni la selección del threshold.